# AI Resume Screening Assistant

This project uses LangChain, RAG, and Generative AI to evaluate resumes against a given Job Description (JD).

The system retrieves relevant information from each resume and generates a structured, evidence-based evaluation covering skills, gaps, candidate summary, strengths, weaknesses, match score, and recruiter review recommendation.

## 1. Project Setup

Load the required environment variables and initialize the OpenAI connection.

In [1]:
import os
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

os.environ.pop("SSL_CERT_FILE", None)

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError("OPENAI_API_KEY was not found.")

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

print("Project setup completed successfully.")

Project setup completed successfully.


## 2. Load Resume PDFs

Load all PDF resumes from the project's `data/resumes` folder using LangChain's PDF document loader.

In [2]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader

DATA_DIR = Path("data")
RESUME_DIR = DATA_DIR / "resumes"

pdf_files = sorted(RESUME_DIR.glob("*.pdf"))

if not pdf_files:
    raise FileNotFoundError("No PDF resumes found in the data/resumes folder.")

documents = []

for pdf_file in pdf_files:
    loader = PyPDFLoader(str(pdf_file))
    documents.extend(loader.load())

print(f"Resumes found: {len(pdf_files)}")
print(f"Total pages loaded: {len(documents)}")

C:\Users\nakul\AppData\Local\Temp\ipykernel_21876\4054400491.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Resumes found: 3
Total pages loaded: 9


## 3. Document Processing

Split the resume documents into smaller overlapping chunks so that relevant information can be retrieved efficiently during the RAG process.

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

chunks = text_splitter.split_documents(documents)

print(f"Total chunks created: {len(chunks)}")

Total chunks created: 30


## 4. Create Embeddings

Convert the resume chunks into numerical embeddings that can be used for semantic search.

In [4]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

print("Embedding model initialized successfully.")

Embedding model initialized successfully.


## 5. Create FAISS Vector Database

Store the resume chunk embeddings in a FAISS vector database for efficient semantic retrieval.

In [5]:
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(documents=chunks, embedding=embeddings)

print("FAISS vector database created successfully.")
print(f"Total vectors stored: {vector_store.index.ntotal}")

FAISS vector database created successfully.
Total vectors stored: 30


## 6. Prepare the Job Description Folder

Create a dedicated folder for Job Description files. Keeping the JD separate from the code makes the application easier to maintain and allows different JDs to be used without changing the Python code.

In [6]:
JOB_DESCRIPTION_DIR = DATA_DIR / "job_descriptions"

JOB_DESCRIPTION_DIR.mkdir(parents=True, exist_ok=True)

print(f"Job Description folder: {JOB_DESCRIPTION_DIR.resolve()}")

Job Description folder: C:\Users\nakul\DataScience\E&ICT_IIT_Guwahati\14_Agentic_AI\Final_Work\ai_resume_screening_assistant\data\job_descriptions


## 7. Locate the Job Description

Locate the existing Job Description file from the project data folder. The notebook reads the file as an input instead of creating or modifying it during execution.

In [7]:
job_description_file = (
    JOB_DESCRIPTION_DIR / "senior_data_scientist_genai_engineer.txt"
)

if not job_description_file.exists():
    raise FileNotFoundError(
        f"Job Description not found: {job_description_file}"
    )

print("Job Description file found successfully.")

Job Description file found successfully.


## 8. Load the Job Description

Read the Job Description from the text file so it can be used as an input to the RAG-based resume evaluation pipeline.

In [8]:
job_description = job_description_file.read_text(encoding="utf-8")

print("Job Description loaded successfully.")
print(f"Characters loaded: {len(job_description)}")

Job Description loaded successfully.
Characters loaded: 1174


## 9. Create Candidate-Specific Retriever

Create a reusable retriever that searches only the resume chunks belonging to the selected candidate. This prevents information from different resumes from being mixed during evaluation.

In [9]:
def get_candidate_retriever(source):
    return vector_store.as_retriever(search_kwargs={"k": 6, "filter": {"source": source}})

print("Candidate-specific retriever created successfully.")

Candidate-specific retriever created successfully.


## 10. Define Retrieval Queries

Use multiple targeted queries to retrieve different types of information from a candidate's resume. This improves the coverage of relevant evidence for the Job Description.

In [10]:
retrieval_queries = [
    "Data Science Machine Learning AI experience years leadership teams",
    "Python Machine Learning Deep Learning NLP Generative AI LLM RAG",
    "AWS Azure GCP vector databases semantic search",
    "production AI ML Docker Kubernetes MLOps CI/CD deployment"
]

print(f"Retrieval queries defined: {len(retrieval_queries)}")

Retrieval queries defined: 4


## 11. Build Candidate Context

Create a reusable function that retrieves relevant resume information using multiple queries and combines the unique results into a single context for evaluation.

In [11]:
def retrieve_candidate_context(source, queries):
    candidate_retriever = get_candidate_retriever(source)

    all_docs = []

    for query in queries:
        docs = candidate_retriever.invoke(query)
        all_docs.extend(docs)

    unique_docs = {}

    for document in all_docs:
        key = (document.metadata.get("source"), document.page_content)
        unique_docs[key] = document

    retrieved_docs = list(unique_docs.values())

    context = "\n\n".join(document.page_content for document in retrieved_docs)

    return retrieved_docs, context

print("Candidate context retrieval function created successfully.")

Candidate context retrieval function created successfully.


## 12. Define Structured Output Schema

Define a Pydantic model to ensure that every resume evaluation follows a consistent structure and includes evidence for each Job Description requirement.

In [12]:
from typing import List
from pydantic import BaseModel, Field

class SkillEvidence(BaseModel):
    requirement: str = Field(description="Job Description requirement being evaluated")
    status: str = Field(description="Evaluation status: Match or Missing")
    evidence: str = Field(description="Short evidence from the resume supporting the status")

class ResumeEvaluation(BaseModel):
    match_score: int = Field(description="Overall match score from 0 to 100")
    matching_skills: List[str] = Field(description="Skills from the resume that match the Job Description")
    missing_skills: List[str] = Field(description="Important Job Description requirements not found in the resume")
    candidate_summary: str = Field(description="Short summary of the candidate's relevant experience")
    strengths: List[str] = Field(description="Key strengths relevant to the Job Description")
    weaknesses: List[str] = Field(description="Relevant gaps based only on the resume")
    hiring_recommendation: str = Field(description="AI-assisted recommendation for recruiter review")
    skill_evidence: List[SkillEvidence] = Field(description="Evidence-based assessment of individual Job Description requirements")

print("Structured output schema created successfully.")

Structured output schema created successfully.


## 13. Create the Pydantic Output Parser

Create a Pydantic output parser that instructs the LLM to return the resume evaluation in the required structured format.

In [13]:
from langchain_core.output_parsers import PydanticOutputParser

output_parser = PydanticOutputParser(pydantic_object=ResumeEvaluation)

format_instructions = output_parser.get_format_instructions()

print("Pydantic output parser created successfully.")

Pydantic output parser created successfully.


## 14. Create the RAG Prompt

Create a prompt that instructs the LLM to evaluate the candidate against the Job Description using only the retrieved resume context and return the required structured output.

In [14]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    """
You are an AI Resume Screening Assistant.

Evaluate the candidate based only on the resume information provided in the context.

Job Description: {job_description}

Resume Context: {context}

Evaluation rules:

1. Evaluate each important Job Description requirement individually.
2. If the resume contains explicit or clearly equivalent evidence, classify the requirement as "Match".
3. Only classify a requirement as "Missing" when there is no supporting evidence in the provided resume context.
4. Do not mark a requirement as missing simply because the exact wording is different.
5. Do not invent or assume experience that is not supported by the resume.
6. For related technologies, use reasonable technical equivalence only when the resume provides clear evidence.
7. For every requirement, provide a short evidence statement from the resume explaining the classification.
8. The match score should reflect the overall alignment between the candidate's documented experience and the Job Description.
9. The hiring recommendation is AI-assisted and must be treated as input for recruiter review, not as an automated final hiring decision.

Important examples:
- EKS or AKS can be treated as evidence of Kubernetes-based technology when clearly stated in the resume.
- Vector knowledge databases can be treated as evidence related to vector databases when clearly stated.
- Related technologies should not be treated as exact matches unless the resume provides reasonable supporting evidence.

Do not use outside knowledge about the candidate.

Provide:
- Matching skills
- Missing skills
- Candidate summary
- Strengths
- Weaknesses
- Match score from 0 to 100
- AI-assisted hiring recommendation for recruiter review
- Requirement-level skill evidence

The skill evidence must contain:
- requirement
- status: "Match" or "Missing"
- evidence

{format_instructions}
"""
)

print("RAG prompt created successfully.")

RAG prompt created successfully.


## 15. Create the Resume Evaluation Function

Create a reusable function that retrieves candidate-specific resume context, sends it along with the Job Description to the LLM, and parses the response into the predefined Pydantic structure.

In [15]:
def evaluate_resume(source, job_description):
    retrieved_docs, context = retrieve_candidate_context(
        source=source, 
        queries=retrieval_queries
        )

    formatted_prompt = prompt.format(
        job_description=job_description,
        context=context,
        format_instructions=format_instructions
    )

    response = llm.invoke(formatted_prompt)

    evaluation = output_parser.parse(response.content)

    return evaluation

print("Resume evaluation function created successfully.")

Resume evaluation function created successfully.


## 16. Identify Candidate Resumes

Create a mapping between each candidate name and the corresponding resume file. This will make candidate evaluation easier to manage and scale.

In [16]:
candidate_resumes = {
    pdf_file.stem: str(pdf_file)
    for pdf_file in pdf_files
}

print("Candidate resumes identified successfully.")

for candidate, source in candidate_resumes.items():
    print(f"- {candidate}")

Candidate resumes identified successfully.
- Eric Smith
- Hoon Y.
- James Howard


## 17. Evaluate Candidates

Evaluate each candidate's resume against the Job Description using the same RAG pipeline and store the structured evaluation results for further analysis.

In [17]:
candidate_evaluations = {}

for candidate, source in candidate_resumes.items():
    candidate_evaluations[candidate] = evaluate_resume(
        source=source,
        job_description=job_description
    )

print("Candidate evaluations completed successfully.")

for candidate in candidate_evaluations:
    print(f"- {candidate}")

Candidate evaluations completed successfully.
- Eric Smith
- Hoon Y.
- James Howard


## 18. Display Candidate Evaluation Summary

Display the key evaluation results for each candidate, including the match score, matching skills, missing skills, and AI-assisted recommendation for recruiter review.

In [18]:
for candidate, evaluation in candidate_evaluations.items():
    print("=" * 80)
    print(f"Candidate: {candidate}")
    print(f"Match Score: {evaluation.match_score}/100")
    print(f"Matching Skills: {', '.join(evaluation.matching_skills)}")
    print(f"Missing Skills: {', '.join(evaluation.missing_skills)}")
    print(f"Recommendation: {evaluation.hiring_recommendation}")

Candidate: Eric Smith
Match Score: 95/100
Matching Skills: 8+ years of experience in Data Science, Machine Learning, or AI, Strong Python programming and data analysis experience, Strong knowledge of Machine Learning and Deep Learning, Experience with Generative AI and Large Language Models (LLMs), Hands-on experience building Retrieval-Augmented Generation (RAG) solutions, Experience with Natural Language Processing (NLP), Experience with at least one major cloud platform: AWS, Azure, or GCP, Experience with vector databases and semantic search, Experience developing production-ready AI/ML solutions, MLOps and CI/CD experience, Experience with Docker or Kubernetes, Experience leading AI/ML projects or teams
Missing Skills: 
Recommendation: Highly recommended for the position due to extensive relevant experience and strong technical skills.
Candidate: Hoon Y.
Match Score: 90/100
Matching Skills: 8+ years of experience in Data Science, Machine Learning, or AI, Strong Python programming 

## 19. Display Requirement-Level Evidence

Display the requirement-level evaluation and supporting resume evidence for each candidate. This provides transparency into how the AI evaluation was generated.

In [19]:
for candidate, evaluation in candidate_evaluations.items():
    print("=" * 80)
    print(f"Candidate: {candidate}")
    
    for item in evaluation.skill_evidence:
        print(f"\nRequirement: {item.requirement}")
        print(f"Status: {item.status}")
        print(f"Evidence: {item.evidence}")

Candidate: Eric Smith

Requirement: 8+ years of experience in Data Science, Machine Learning, or AI
Status: Match
Evidence: 15 years of hands-on professional experience as a Data Scientist & AI/ML Data Architect.

Requirement: Strong Python programming and data analysis experience
Status: Match
Evidence: Several Statistical languages and tools including Python, SAS, Rapid Miner.

Requirement: Strong knowledge of Machine Learning and Deep Learning
Status: Match
Evidence: Advanced modeling techniques, including Neural Networks and Deep Learning for NLP.

Requirement: Experience with Generative AI and Large Language Models (LLMs)
Status: Match
Evidence: Expert in NLP/NLU, various Gen AI technologies including multi-modal embedding, Vector knowledge databases, RAG Agentic solutions and fine-tuning of LLMs.

Requirement: Hands-on experience building Retrieval-Augmented Generation (RAG) solutions
Status: Match
Evidence: Gen AI Solution Architect and Lead Data Scientist for a European Automob

## 20. Create Candidate Evaluation Table

Convert the structured evaluation results into a tabular format for easier comparison of candidates and their key screening results.

In [20]:
import pandas as pd

evaluation_summary = []

for candidate, evaluation in candidate_evaluations.items():
    evaluation_summary.append({
        "Candidate": candidate,
        "Match Score": evaluation.match_score,
        "Matching Skills": len(evaluation.matching_skills),
        "Missing Skills": len(evaluation.missing_skills),
        "Recommendation": evaluation.hiring_recommendation
    })

evaluation_df = pd.DataFrame(evaluation_summary)

evaluation_df

,Candidate,Match Score,Matching Skills,Missing Skills,Recommendation
0,Eric Smith,95,12,0,Highly recommended for the position due to ext...
1,Hoon Y.,90,9,3,"Strong candidate for the role, with significan..."
2,James Howard,95,12,0,Highly recommended for the Senior Data Scienti...


## 21. Add Candidate Summaries

Create a concise candidate-level view containing the summary, strengths, weaknesses, and AI-assisted recommendation generated from the resume evaluation.

In [21]:
for candidate, evaluation in candidate_evaluations.items():
    print("=" * 80)
    print(f"Candidate: {candidate}")
    print(f"\nSummary:\n{evaluation.candidate_summary}")
    print(f"\nStrengths:\n- " + "\n- ".join(evaluation.strengths))
    print(f"\nWeaknesses:\n- " + "\n- ".join(evaluation.weaknesses))
    print(f"\nRecommendation:\n{evaluation.hiring_recommendation}")

Candidate: Eric Smith

Summary:
The candidate has over 15 years of experience in Data Science and AI, with a strong focus on Generative AI, NLP, and cloud technologies. They have led AI/ML projects and possess extensive knowledge in machine learning and deep learning techniques.

Strengths:
- Extensive experience in AI/ML and Data Science with a focus on Generative AI
- Strong programming skills in Python and experience with various cloud platforms
- Proven track record of leading AI/ML projects and teams
- Hands-on experience with RAG solutions and NLP technologies

Weaknesses:
- 

Recommendation:
Highly recommended for the position due to extensive relevant experience and strong technical skills.
Candidate: Hoon Y.

Summary:
The candidate is a senior ML leader with extensive experience in building and deploying production AI/ML systems across federal and commercial domains. They have a proven track record in defining ML strategies and leading teams, with notable achievements in gener

## 22. Compare Candidates by Job Requirements

Create a requirement-level comparison showing whether each candidate has evidence supporting each Job Description requirement.

In [22]:
requirement_comparison = {}

for candidate, evaluation in candidate_evaluations.items():
    for item in evaluation.skill_evidence:
        if item.requirement not in requirement_comparison:
            requirement_comparison[item.requirement] = {}

        requirement_comparison[item.requirement][candidate] = item.status

comparison_df = pd.DataFrame.from_dict(requirement_comparison, orient="index")

comparison_df.index.name = "Job Requirement"

comparison_df

,Eric Smith,Hoon Y.,James Howard
Job Requirement,,,
"8+ years of experience in Data Science, Machine Learning, or AI",Match,Match,Match
Strong Python programming and data analysis experience,Match,Match,Match
Strong knowledge of Machine Learning and Deep Learning,Match,Match,Match
Experience with Generative AI and Large Language Models (LLMs),Match,Match,Match
Hands-on experience building Retrieval-Augmented Generation (RAG) solutions,Match,Match,Match
Experience with Natural Language Processing (NLP),Match,Match,Match
"Experience with at least one major cloud platform: AWS, Azure, or GCP",Match,Match,Match
Experience with vector databases and semantic search,Match,Match,Match
Experience developing production-ready AI/ML solutions,Match,Match,Match


## 23. Compare Supporting Evidence

Create a comparison table containing the evidence generated for each candidate against each Job Description requirement. This allows recruiters to review the basis of the AI evaluation.

In [23]:
evidence_comparison = {}

for candidate, evaluation in candidate_evaluations.items():
    for item in evaluation.skill_evidence:
        if item.requirement not in evidence_comparison:
            evidence_comparison[item.requirement] = {}

        evidence_comparison[item.requirement][candidate] = (
            f"{item.status}: {item.evidence}"
        )

evidence_df = pd.DataFrame.from_dict(
    evidence_comparison,
    orient="index"
)

evidence_df.index.name = "Job Requirement"

evidence_df

,Eric Smith,Hoon Y.,James Howard
Job Requirement,,,
"8+ years of experience in Data Science, Machine Learning, or AI",Match: 15 years of hands-on professional exper...,Match: Senior ML leader building and shipping ...,Match: With 20+ years of experience and a PhD ...
Strong Python programming and data analysis experience,Match: Several Statistical languages and tools...,Match: Led the design and deployment of RAG an...,Match: Writing scripts in Python and developin...
Strong knowledge of Machine Learning and Deep Learning,"Match: Advanced modeling techniques, including...",Match: Owned ML strategy for creative understa...,"Match: Designed, architected, and led enterpri..."
Experience with Generative AI and Large Language Models (LLMs),"Match: Expert in NLP/NLU, various Gen AI techn...",Match: Led the design and deployment of RAG an...,Match: Deployed scalable Generative AI archite...
Hands-on experience building Retrieval-Augmented Generation (RAG) solutions,Match: Gen AI Solution Architect and Lead Data...,Match: Led the design and deployment of RAG an...,Match: Architected scalable RAG pipelines leve...
Experience with Natural Language Processing (NLP),Match: Developed NLP based customer call cente...,Match: Led development of multimodal CV+NLP an...,Match: Utilized Cloud Natural Language and bui...
"Experience with at least one major cloud platform: AWS, Azure, or GCP","Match: Cloud technologies: AWS, Azure, GCP, Sp...",Match: Evaluate and select platforms and tooli...,"Match: Worked extensively with AWS, Azure, and..."
Experience with vector databases and semantic search,Match: Expert in various Gen AI technologies i...,Match: Evaluate and select platforms and tooli...,Match: Implemented RAG pipelines using vector ...
Experience developing production-ready AI/ML solutions,Match: Created enterprise level data migration...,Match: Proven track record defining end-to-end...,Match: Delivered production-grade AI systems i...


## 24. Validate Structured Evaluations

Validate the generated candidate evaluations to ensure that match scores are within the expected range and that the required structured fields are populated.

In [24]:
validation_results = []

for candidate, evaluation in candidate_evaluations.items():
    validation_results.append({
        "Candidate": candidate,
        "Score Valid": 0 <= evaluation.match_score <= 100,
        "Summary Present": bool(evaluation.candidate_summary.strip()),
        "Evidence Present": len(evaluation.skill_evidence) > 0,
        "Recommendation Present": bool(evaluation.hiring_recommendation.strip())
    })

validation_df = pd.DataFrame(validation_results)

validation_df

,Candidate,Score Valid,Summary Present,Evidence Present,Recommendation Present
0,Eric Smith,True,True,True,True
1,Hoon Y.,True,True,True,True
2,James Howard,True,True,True,True


## 25. Validate Requirement Statuses

Check that every requirement-level evaluation uses a valid status of either "Match" or "Missing". This helps prevent unexpected values from entering the final results.

In [25]:
valid_statuses = {"Match", "Missing"}

status_validation = []

for candidate, evaluation in candidate_evaluations.items():
    invalid_statuses = []

    for item in evaluation.skill_evidence:
        if item.status not in valid_statuses:
            invalid_statuses.append(item.status)

    status_validation.append({
        "Candidate": candidate,
        "All Statuses Valid": len(invalid_statuses) == 0
    })

status_validation_df = pd.DataFrame(status_validation)

status_validation_df

,Candidate,All Statuses Valid
0,Eric Smith,True
1,Hoon Y.,True
2,James Howard,True


## 26. Validate Missing-Skill Evidence

Review requirements classified as "Missing" and compare them with the retrieved candidate context. This helps identify potential retrieval or evaluation errors before the results are used by a recruiter.

In [26]:
grounding_checks = []

for candidate, source in candidate_resumes.items():
    _, context = retrieve_candidate_context(
        source=source,
        queries=retrieval_queries
    )

    evaluation = candidate_evaluations[candidate]

    missing_requirements = [
        item.requirement
        for item in evaluation.skill_evidence
        if item.status == "Missing"
    ]

    grounding_checks.append({
        "Candidate": candidate,
        "Missing Requirements": len(missing_requirements),
        "Retrieved Context Available": len(context) > 0
    })

grounding_check_df = pd.DataFrame(grounding_checks)

grounding_check_df

,Candidate,Missing Requirements,Retrieved Context Available
0,Eric Smith,0,True
1,Hoon Y.,3,True
2,James Howard,0,True
